# Sensitivity to the weather lag window

Compare 1–30 weather input days for the hybrid model and the air-temperature-only
model. A six-day input window includes the current day and five preceding days.

Each member's final weights are evaluated on the original chronological training
and validation splits with its baseline rebuilt by native inference. The notebook
recomputes the native Poisson loss components on the reconstructed predictions.
Daily panels show the national daily total term. Weekly panels show the weekly
national total term, which aggregates age and retains sex. The table also
retains the other native components. These are deterministic final-checkpoint
reconstruction losses. The training-time objective also depends on the fitted
baseline state, batch-normalization mode, augmentation, and epoch weighting.

The hybrid sweep uses air temperature and dew point and the weather-only sweep
uses air temperature. Compare changes with lag within each setup. Full execution
evaluates 600 checkpoints and is intended for HPC.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys

cwd = Path.cwd()
repo = next(
    (
        path for path in (cwd, *cwd.parents)
        if (path / "configs" / "training.example.yaml").is_file()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError("Open the notebook from the sari-weather-mortality checkout.")

sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "src"))

import matplotlib.pyplot as plt
import pandas as pd

# Load helpers from this checkout even if Jupyter cached another notebooks package.
utils_path = repo / "notebooks" / "utils.py"
spec = spec_from_file_location("sari_notebook_utils", utils_path)
notebook_utils = module_from_spec(spec)
spec.loader.exec_module(notebook_utils)

ensemble = notebook_utils.ensemble
DEVICE = notebook_utils.DEVICE

from sari_t.aggregation import aggregate_mortality
from sari_t.inference import InferenceModel
from sari_t.run.checkpoint import load_checkpoint, load_run_config, restore_predictor
from sari_t.training.loss import calculate_loss_components

## Load models for inspection

The next cell loads member 000 for the selected window from each setup onto CPU.
Inspect `model` for the hybrid predictor or `models[setup]` for another predictor.

In [ ]:
inspection_member = 0
inspection_window = 6
setups = {
    "Hybrid (temperature + dew point)": "temperature_kernel_hybrid",
    "Weather only (temperature)": "temperature_kernel_weather_only",
}
windows = range(1, 31)
sweeps = {}
for label, group in setups.items():
    runs = {}
    for run in sorted((repo / "runtime" / "training" / "sets" / group).iterdir()):
        run = run.resolve()
        config = load_run_config(run / "config.yaml")
        lag = config.model_config.kernel_T_days
        if lag in runs:
            raise ValueError(f"Multiple {lag}-day runs in {group}")
        paths = [run / "checkpoints" / f"final_{member:03d}.pt" for member in range(10)]
        if not all(path.is_file() for path in paths):
            raise FileNotFoundError(f"Incomplete ensemble for {run}")
        runs[lag] = (config, paths)
    if set(runs) != set(windows):
        raise ValueError(f"{group} must contain one run for each window in {list(windows)}")
    sweeps[label] = runs

models = {
    label: restore_predictor(
        load_checkpoint(runs[inspection_window][1][inspection_member]), device="cpu",
    ).eval()
    for label, runs in sweeps.items()
}
model = models["Hybrid (temperature + dew point)"]
model

## Recompute training and validation loss

In [ ]:
from datetime import timedelta

def split_checkpoints(workspace, result):
    period = workspace.period
    training_end = period.start_date + timedelta(days=training_days - 1)
    validation_start = training_end + timedelta(days=1)
    losses = {}
    for split, start, end in (
        ("Training", period.start_date, training_end),
        ("Validation", validation_start, period.end_date),
    ):
        offset = (start - period.start_date).days
        count = (end - start).days + 1
        counts = result.predictions.total_deaths[offset:offset + count]
        labels = select_mortality_labels(result.labels, PredictionPeriod(start, end))
        terms = calculate_loss_components(aggregate_mortality(counts), labels)
        losses.update({(split, term): loss for term, loss in terms.items()})
    return losses

from sari_t.data.training_inputs import select_mortality_labels
from sari_t.models.config import PredictionPeriod

rows = []
expected_split = None
for setup, runs in sweeps.items():
    for lag in windows:
        config, paths = runs[lag]
        weeks = ((config.period.end_date - config.period.start_date).days + 1) // 7
        validation_weeks = int(weeks * config.training_config.validation_fraction)
        if not 0 < validation_weeks < weeks:
            raise ValueError(f"{setup}, {lag} days has no usable training/validation split")
        training_days = 7 * (weeks - validation_weeks)
        split_key = (config.period, training_days)
        if expected_split is None:
            expected_split = split_key
        elif split_key != expected_split:
            raise ValueError("All lag runs must use the same training and validation dates.")

        print(f"{setup}, {lag} input days", flush=True)
        inference_model = InferenceModel.from_checkpoint(
            paths[0], data_config=config.data_config, period=config.period, device=DEVICE,
        )
        values = ensemble(inference_model, paths, split_checkpoints, keep_members=True)
        for (split, term), losses in values.items():
            rows.extend((setup, lag, member, split, term, loss)
                        for member, loss in enumerate(losses.tolist()))
        del inference_model

loss_by_member = pd.DataFrame(rows, columns=["Setup", "Window (days)", "Member",
                                              "Split", "Loss term", "Loss"])
loss_by_window = loss_by_member.groupby(["Setup", "Window (days)", "Split", "Loss term"])[
    "Loss"
].agg(["mean", "min", "max"]).reset_index()
loss_by_window

## Daily and weekly loss across lag windows

Lines are ensemble means. Shading spans the lowest and highest member losses.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for row, (term, frequency) in enumerate((("daily_total", "Daily"), ("weekly_total", "Weekly"))):
    for col, split in enumerate(("Training", "Validation")):
        ax = axes[row, col]
        selected = loss_by_window.query("`Loss term` == @term and Split == @split")
        for setup, group in selected.groupby("Setup", sort=False):
            group = group.sort_values("Window (days)")
            x = group["Window (days)"].to_numpy()
            line, = ax.plot(x, group["mean"], label=setup)
            ax.fill_between(x, group["min"], group["max"], color=line.get_color(), alpha=0.2)
        ax.axvline(6, color="0.5", linestyle=":", linewidth=1)
        ax.set_title(f"{split} · {frequency.lower()}")
        ax.grid(alpha=0.2)
        if col == 0:
            ax.set_ylabel("Summed Poisson loss")
        if row == 1:
            ax.set_xlabel("Weather input window (days)")
axes[0, 0].legend(frameon=False)
fig.tight_layout()
plt.show()